## Creating a Unified Dataset for YOLOv8 Training

In [ ]:
import os
import shutil
import yaml

# 1. Configuration & Aliases

DATA_ROOT = r'C:/id_card/Data' # Base folder

datasets = {
    'Detection': os.path.join(DATA_ROOT, 'Detection.v1i.yolov8'),
    'DL': os.path.join(DATA_ROOT, 'Driving License project.v1i.yolov8')
}

yaml_files = {
    'Detection': 'data_2.yaml',
    'DL': 'data_3.yaml'
}

UNIFIED_DIR = os.path.join(DATA_ROOT, 'Unified_Dataset_2_3_v2')
splits = ['train', 'valid', 'test']

# Define aliases to merge duplicate classes
CLASS_ALIASES = {
    'aadhar-no': 'aadhaar number',
    'add': 'address',
    'father name': 'father'
}

# 
CARD_TRIGGERS = {
    'aadhaar number': 'aadhar_card',
    'pan number': 'pan_card',
    'dl_no': 'dl_card'
}

# 2. Build the Unified Class List
all_classes = set()
dataset_class_maps = {} 

# First pass: Get all existing classes
for ds_name, ds_path in datasets.items():
    yaml_path = os.path.join(ds_path, yaml_files[ds_name])
    
    if os.path.exists(yaml_path):
        with open(yaml_path, 'r') as f:
            data = yaml.safe_load(f)
            
        names = data['names']
        dataset_class_maps[ds_name] = {}
        
        for i, name in enumerate(names):
            canonical_name = CLASS_ALIASES.get(name, name)
            dataset_class_maps[ds_name][i] = canonical_name
            all_classes.add(canonical_name)

# Add our new full-card classes to the master list
for card_class in CARD_TRIGGERS.values():
    all_classes.add(card_class)

# Sort for consistency and assign new IDs
all_classes = sorted(list(all_classes))
unified_class_to_id = {name: i for i, name in enumerate(all_classes)}

print(f"Total Unique Classes (Including Full-Card Classes): {len(all_classes)}")
print("Unified Class List:", all_classes)

# Create translation map
id_translation_map = {}
for ds_name, class_map in dataset_class_maps.items():
    id_translation_map[ds_name] = {}
    for old_id, canonical_name in class_map.items():
        id_translation_map[ds_name][old_id] = unified_class_to_id[canonical_name]

# 3. Create Unified Directory Structure

if os.path.exists(UNIFIED_DIR):
    shutil.rmtree(UNIFIED_DIR)

for split in splits:
    os.makedirs(os.path.join(UNIFIED_DIR, split, 'images'), exist_ok=True)
    os.makedirs(os.path.join(UNIFIED_DIR, split, 'labels'), exist_ok=True)


# 4. Process, Merge, and Inject Labels

for ds_name, ds_path in datasets.items():
    print(f"\nProcessing {ds_name} dataset...")
    
    for split in splits:
        img_dir = os.path.join(ds_path, split, 'images')
        lbl_dir = os.path.join(ds_path, split, 'labels')
        
        if not os.path.exists(img_dir) or not os.path.exists(lbl_dir):
            continue
            
        images = [f for f in os.listdir(img_dir) if f.endswith(('.jpg', '.jpeg', '.png'))]
        
        for img_name in images:
            unique_prefix = f"{ds_name}_"
            new_img_name = unique_prefix + img_name
            lbl_name = os.path.splitext(img_name)[0] + '.txt'
            new_lbl_name = unique_prefix + lbl_name
            
            src_img = os.path.join(img_dir, img_name)
            dst_img = os.path.join(UNIFIED_DIR, split, 'images', new_img_name)
            
            src_lbl = os.path.join(lbl_dir, lbl_name)
            dst_lbl = os.path.join(UNIFIED_DIR, split, 'labels', new_lbl_name)
            
            shutil.copy(src_img, dst_img)
            
            # Map old IDs and check for card triggers
            cards_to_add = set()
            
            if os.path.exists(src_lbl):
                with open(src_lbl, 'r') as f_in, open(dst_lbl, 'w') as f_out:
                    for line in f_in:
                        parts = line.strip().split()
                        if len(parts) >= 5:
                            old_id = int(parts[0])
                            
                            # Skip if old_id is not in our translation map for some reason
                            if old_id not in id_translation_map[ds_name]:
                                continue
                                
                            new_id = id_translation_map[ds_name][old_id]
                            canonical_name = all_classes[new_id]
                            
                            # Check if this class triggers a full-card injection
                            if canonical_name in CARD_TRIGGERS:
                                cards_to_add.add(CARD_TRIGGERS[canonical_name])
                                
                            # Write the original (now remapped) bounding box
                            new_line = f"{new_id} {' '.join(parts[1:])}\n"
                            f_out.write(new_line)
                            
                    # After writing all existing boxes, inject the full-card box(es)
                    # Coordinates: x_center=0.5, y_center=0.5, width=1.0, height=1.0
                    for card_class in cards_to_add:
                        card_id = unified_class_to_id[card_class]
                        f_out.write(f"{card_id} 0.5 0.5 1.0 1.0\n")

print("\nFiles successfully merged, labels remapped, and full-card boxes injected!")


# 5. Generate Unified YAML

unified_yaml_path = os.path.join(UNIFIED_DIR, 'unified_data.yaml')
unified_yaml_content = {
    'train': '../train/images',
    'val': '../valid/images',
    'test': '../test/images',
    'nc': len(all_classes),
    'names': all_classes
}

with open(unified_yaml_path, 'w') as f:
    yaml.dump(unified_yaml_content, f, sort_keys=False)

print(f"Unified YAML created at: {unified_yaml_path}")

Total Unique Classes (Including Full-Card Classes): 26
Unified Class List: ['aadhaar number', 'aadhar_card', 'address', 'blood_group', 'dl_card', 'dl_no', 'dob', 'emblem', 'father', 'gender', 'goi symbol', 'issue date', 'logo', 'name', 'pan number', 'pan_card', 'pancard', 'photo', 'relation_with', 'rto', 'state', 'uiai icon', 'uiai symbol', 'vehicle_type', 'vid', 'yob']

Processing Detection dataset...

Processing DL dataset...

Files successfully merged, labels remapped, and full-card boxes injected!
Unified YAML created at: C:/id_card/Data\Unified_Dataset_2_3_v2\unified_data.yaml
